# Lab 04-01: Build and Run a Minimal Coding Agent

**Book chapter:** Chapter 04: Using and Customizing Coding Agents  
**Estimated time:** 30 minutes

## Overview

Cymbal Retail's platform team is preparing to roll out coding agents such as
Antigravity and Claude Code. Before writing rules and hooks for them, the architects
want to understand what actually happens inside a coding agent: how the model, the
tools, and the execution loop fit together, and where controls such as a sandbox and a
tool allowlist apply.

In this lab, you build a minimal coding agent with Gemini function calling, run it on
a project whose tests fail, watch each step of its reason-act-observe loop, verify its
work independently, and confirm that its file tools cannot leave the workspace.

## Learning Objectives

By the end of this lab, you will be able to:

- Explain how a model, tools, and a loop combine into a coding agent
- Define agent tools as Python functions with type hints and docstrings
- Observe a reason-act-observe loop step by step
- Verify an agent's work independently instead of trusting its summary
- Describe how a workspace boundary and a step limit contain an agent

## Before You Begin

- Complete **Lab 00: Set Up the Lab Environment** in the `Appendix-C-Setup` folder

**Cost:** A few cents for Gemini requests. The lab creates no Google Cloud resources.

## How to Use This Notebook

Run the cells in order, from top to bottom, by selecting each code cell and pressing **Shift+Enter**. Read the explanation above each cell before you run it, and compare your output with the expected result that the text describes.

## Key Concepts

| Concept | Meaning |
|---|---|
| **Tool** | A function the model can ask the agent to run, such as reading a file or running tests |
| **Reason-act-observe loop** | The model decides on an action, the agent runs it, and the result goes back to the model |
| **Sandbox boundary** | A limit on what tools can reach; here, file tools stay inside one folder |
| **Step limit** | A maximum number of loop iterations that stops runaway agents |

> **Note:** The agent in this lab is deliberately small so that every part is visible.
> Production coding agents add planning, context management, OS-level sandboxing, and
> approval flows, but they follow the same loop.

## Setup

In [ ]:
%pip install --quiet --upgrade google-genai

> **Note:** If you are using Google Colab and a message asks you to restart the
> session after the installation, click **Restart session**, and then continue
> from the next cell. You don't need to run the installation cell again.

In [ ]:
import os

# Replace with your lab project ID (see Appendix C).
PROJECT_ID = "your-project-id"  # @param {type:"string"}
# A current Gemini Flash model ID. Check the Google models page if this
# model has been retired.
MODEL = "gemini-3.8-flash"  # @param {type:"string"}

# Fall back to the environment variables set by labenv.sh (Appendix C).
if PROJECT_ID == "your-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
MODEL = os.environ.get("GEMINI_MODEL", MODEL)
assert PROJECT_ID != "your-project-id", "Set PROJECT_ID to your project ID."
# The agent code reads these environment variables.
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GEMINI_MODEL"] = MODEL
print("Project:", PROJECT_ID)
print("Model:  ", MODEL)

In [ ]:
import sys

if "google.colab" in sys.modules:
    # In Colab, sign in with the Google account that owns the lab project.
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Signed in to Google Cloud from Colab.")
else:
    # In Jupyter, VS Code, or Workbench, the client libraries use Application
    # Default Credentials. If a later cell reports a credentials error, run
    # this command in a terminal and then rerun the cell:
    #   gcloud auth application-default login
    print("Using Application Default Credentials.")

# Part 1: Build the Coding Agent

Run the following cells to define the agent. Read the explanation above each cell; the
same code is printed in the book as coding_agent.py.

### The Gemini Client and Helpers

This cell creates the Gemini client, defines the base instruction that every agent
run starts from, and defines `run`, a helper that runs a command and returns the end of
its output. The project and model come from the environment variables set during setup.

In [ ]:
import json
import os
import pathlib
import re
import subprocess
import sys

from google import genai
from google.genai import types

PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]
MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")
client = genai.Client(vertexai=True, project=PROJECT, location="global")

BASE_INSTRUCTION = """You are a coding agent working in a small Python
project. Use the tools to inspect files, make changes, and run the
tests. Make the smallest change that completes the task, run the tests
before you finish, and end with a short summary of what you changed."""


def run(command, cwd, shell=False):
    """Runs a command and returns the end of its combined output."""
    done = subprocess.run(command, cwd=cwd, shell=shell, text=True,
                          capture_output=True, timeout=120)
    return (done.stdout + done.stderr)[-3000:] or "(no output)"

### The Tools

Each tool is an ordinary Python function. The Gen AI SDK turns the function's name,
type hints, and docstring into a **tool declaration**, so the docstring is what the model
reads when it decides which tool to call.

The `inside` function resolves every path and refuses any path outside the workspace.
This is a simple **sandbox boundary**: whatever the model asks for, the file tools cannot
read or write anything else on your computer.

In [ ]:
def make_tools(root, skills):
    """Creates the agent's tools. File tools stay inside the workspace."""

    def inside(path):
        full = (root / path).resolve()
        if not full.is_relative_to(root):
            raise PermissionError(f"{path} is outside the workspace")
        return full

    def list_files() -> list[str]:
        """Lists the files in the workspace."""
        return sorted(p.relative_to(root).as_posix()
                      for p in root.rglob("*")
                      if p.is_file() and "__pycache__" not in p.parts)

    def read_file(path: str) -> str:
        """Returns the text of a workspace file.

        Args:
            path: The file path, relative to the workspace root.
        """
        return inside(path).read_text()

    def write_file(path: str, content: str) -> str:
        """Creates or replaces a workspace file.

        Args:
            path: The file path, relative to the workspace root.
            content: The complete new content of the file.
        """
        target = inside(path)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content)
        return f"Wrote {len(content)} characters to {path}."

    def run_tests() -> str:
        """Runs the project's unit tests and returns the report."""
        return run([sys.executable, "-m", "unittest", "discover", "-v"],
                   root)

    def run_command(command: str) -> str:
        """Runs a shell command in the workspace and returns its output.

        Args:
            command: The shell command to run.
        """
        return run(command, root, shell=True)

    def load_skill(name: str) -> str:
        """Returns the full instructions of an available skill.

        Args:
            name: The name of the skill to load.
        """
        return skills[name]["body"]

    return [list_files, read_file, write_file, run_tests, run_command,
            load_skill]

### Rules, Skills, and Hooks

These functions load the customizations from the same locations that Antigravity uses:

| Function | Customization | Location |
|---|---|---|
| `load_rules` | Rules added to every run | `AGENTS.md` |
| `load_skills` | Skills, shown by name and description until loaded | `.agents/skills/<name>/SKILL.md` |
| `pre_tool_use` | Hooks that can deny a tool call | `.agents/hooks.json` |

A hook receives the event as JSON on standard input and prints a JSON decision. Because
it is ordinary code, it enforces policy the same way every time.

In [ ]:
def front_matter(text):
    """Splits a Markdown file into frontmatter fields and a body."""
    match = re.match(r"---\n(.*?)\n---\n(.*)", text, re.S)
    if not match:
        return {}, text
    fields = dict(re.findall(r"^(\w+):\s*(.*)$", match.group(1), re.M))
    return fields, match.group(2).strip()


def load_rules(root):
    """Returns the project rules in AGENTS.md, if the file exists."""
    rules = root / "AGENTS.md"
    return rules.read_text() if rules.exists() else ""


def load_skills(root):
    """Finds the skills in .agents/skills/<name>/SKILL.md."""
    skills = {}
    for path in sorted(root.glob(".agents/skills/*/SKILL.md")):
        fields, body = front_matter(path.read_text())
        name = fields.get("name", path.parent.name)
        skills[name] = {"description": fields.get("description", ""),
                        "body": body}
    return skills


def pre_tool_use(root, tool, args):
    """Runs matching PreToolUse hooks and returns a reason if one denies."""
    config = root / ".agents" / "hooks.json"
    if not config.exists():
        return None
    event = json.dumps({"event": "PreToolUse", "tool_name": tool,
                        "tool_input": args})
    for group in json.loads(config.read_text()).values():
        for entry in group.get("PreToolUse", []):
            if not re.fullmatch(entry.get("matcher", ".*"), tool):
                continue
            for hook in entry["hooks"]:
                # Run Python hooks with the same interpreter as the agent.
                command = hook["command"]
                if command.startswith("python "):
                    command = f'"{sys.executable}" {command[7:]}'
                done = subprocess.run(
                    command, cwd=root, shell=True, input=event, text=True,
                    capture_output=True, timeout=hook.get("timeout", 30))
                decision = json.loads(done.stdout or "{}")
                if decision.get("decision") == "deny":
                    return decision.get("reason", "Denied by a hook.")
    return None

### The Reason-Act-Observe Loop

`run_agent` is the heart of a coding agent:

1. It builds the instruction from the base instruction, the rules, and the skill catalog.
2. It sends the conversation to Gemini, which either replies with text or asks for tools.
3. For each tool call, it runs the PreToolUse hooks, checks the tool allowlist, runs the
   tool, and records the result.
4. It sends the results back and repeats until the model replies with text or the step
   limit is reached.

Automatic function calling is turned off so that every step is visible in the activity log.

In [ ]:
def brief(value, width=72):
    """Shortens a value to one line for the activity log."""
    text = value if isinstance(value, str) else json.dumps(value)
    text = " ".join(text.split())
    return text if len(text) <= width else text[:width - 3] + "..."


def run_agent(task, workspace, system=None, allowed_tools=None,
              max_steps=20):
    """Runs the reason-act-observe loop until the model stops calling."""
    root = pathlib.Path(workspace).resolve()
    skills = load_skills(root)
    tools = {tool.__name__: tool for tool in make_tools(root, skills)}
    if allowed_tools is not None:
        tools = {n: t for n, t in tools.items() if n in allowed_tools}

    instruction = system or BASE_INSTRUCTION
    rules = load_rules(root)
    if rules:
        instruction += "\n\nProject rules from AGENTS.md:\n" + rules
    if skills and "load_skill" in tools:
        catalog = "\n".join(f"- {name}: {skill['description']}"
                            for name, skill in skills.items())
        instruction += ("\n\nAvailable skills. Call load_skill before "
                        "starting a task that matches one:\n" + catalog)

    config = types.GenerateContentConfig(
        system_instruction=instruction, tools=list(tools.values()),
        automatic_function_calling=types.AutomaticFunctionCallingConfig(
            disable=True))
    contents = [types.Content(role="user",
                              parts=[types.Part.from_text(text=task)])]

    for step in range(1, max_steps + 1):
        response = client.models.generate_content(
            model=MODEL, contents=contents, config=config)
        contents.append(response.candidates[0].content)
        if not response.function_calls:
            print(f"\nAgent: {response.text}")
            return response.text

        replies = []
        for call in response.function_calls:
            args = dict(call.args or {})
            reason = pre_tool_use(root, call.name, args)
            if reason:
                result = {"error": f"Blocked by a hook: {reason}"}
            elif call.name not in tools:
                result = {"error": f"The tool {call.name} is not allowed."}
            else:
                try:
                    result = {"result": tools[call.name](**args)}
                except Exception as error:  # Report errors to the model.
                    result = {"error": str(error)}
            print(f"[{step}] {call.name}({brief(args, 40)}) -> "
                  f"{brief(result)}")
            replies.append(types.Part.from_function_response(
                name=call.name, response=result))
        contents.append(types.Content(role="user", parts=replies))

    print("Stopped: the step limit was reached.")
    return None

### Subagents

A subagent is defined in a Markdown file. Its frontmatter `tools` list becomes a **tool
allowlist**, and its body becomes the system instruction. The subagent starts with a fresh
conversation, so it does not inherit the main agent's history.

In [ ]:
def run_subagent(name, task, workspace):
    """Runs a subagent defined in .agents/agents/<name>.md."""
    root = pathlib.Path(workspace).resolve()
    path = root / ".agents" / "agents" / f"{name}.md"
    fields, body = front_matter(path.read_text())
    tools = re.findall(r"\w+", fields.get("tools", ""))
    print(f"Delegating to the {name} subagent with tools: {tools}")
    return run_agent(task, root, system=body, allowed_tools=tools)

# Part 2: Run the Agent on a Failing Project

## Step 1: Create a small project whose tests fail

The project has one function, `apply_discount`, that is not implemented yet, and four unit tests that define the expected behavior.

In [ ]:
import pathlib
import sys
import textwrap


WORKSPACE = pathlib.Path("workspace-pricing").resolve()
WORKSPACE.mkdir(exist_ok=True)

(WORKSPACE / "pricing.py").write_text(textwrap.dedent('''\
    """Pricing rules for Cymbal Retail."""


    def apply_discount(price, tier):
        """Returns the price after the loyalty discount for a tier."""
        raise NotImplementedError
    '''))

(WORKSPACE / "test_pricing.py").write_text(textwrap.dedent('''\
    import unittest

    from pricing import apply_discount


    class ApplyDiscountTest(unittest.TestCase):
        def test_standard_members_pay_full_price(self):
            self.assertEqual(apply_discount(100.0, "standard"), 100.0)

        def test_silver_members_get_five_percent_off(self):
            self.assertEqual(apply_discount(100.0, "silver"), 95.0)

        def test_gold_members_get_ten_percent_off(self):
            self.assertEqual(apply_discount(80.0, "gold"), 72.0)

        def test_unknown_tier_is_rejected(self):
            with self.assertRaises(ValueError):
                apply_discount(100.0, "platinum")
    '''))
print("Created the project in", WORKSPACE)

## Step 2: Record a baseline by running the tests yourself

Before the agent changes anything, record a baseline by running the tests yourself.

In [ ]:
TEST = [sys.executable, "-m", "unittest", "discover"]
print(run(TEST, WORKSPACE))

**Expected result:** all four tests fail with `NotImplementedError`.

## Step 3: Ask the agent to fix the project, and watch each step

Give the agent the task. Each numbered line in the output is one tool call: the step number, the tool and its arguments, and a shortened result.

In [ ]:
run_agent("Implement apply_discount in pricing.py so that every test in "
          "test_pricing.py passes. Do not change the tests.", WORKSPACE)

**Expected result:** the agent lists and reads the files, writes an implementation, runs the tests, and ends with a short summary.

## Step 4: Verify the result independently

Never rely on the agent's own summary. Run the tests yourself and read the code it wrote.

In [ ]:
print(run(TEST, WORKSPACE))
print((WORKSPACE / "pricing.py").read_text())

**Expected result:** `OK` for all four tests, followed by the new implementation of `apply_discount`.

## Step 5: Confirm that the file tools cannot leave the workspace

Call the `read_file` tool directly with a path outside the workspace.

In [ ]:
read_file = {t.__name__: t for t in make_tools(WORKSPACE, {})}["read_file"]
try:
    read_file("../coding_agent.py")
except PermissionError as error:
    print("Blocked:", error)

**Expected result:** `Blocked: ../coding_agent.py is outside the workspace`.

## Step 6: Experiment With the Step Limit

Run the cell for Step 1 again to restore the unimplemented function, and then run the
next cell. With `max_steps=2`, the agent stops before it can finish the task, which shows
how a step limit protects against runaway loops.

In [ ]:
run_agent("Implement apply_discount in pricing.py so that every test in "
          "test_pricing.py passes. Do not change the tests.", WORKSPACE,
          max_steps=2)

**Expected result:** a few tool calls, followed by `Stopped: the step limit was reached.`

## Check Your Understanding

**1. What does the model see when it decides which tool to call?**

<details><summary>Show answer</summary>

The tool declarations that the SDK builds from each function's name, type hints, and docstring, together with the instruction and the conversation so far.

</details>

**2. Why does the lab run the tests itself after the agent says it is done?**

<details><summary>Show answer</summary>

An agent's summary can be wrong. Independent verification by tests and human review is the most important control when agents change code.

</details>

**3. What stops the file tools from reading files elsewhere on your computer?**

<details><summary>Show answer</summary>

The inside function resolves each path and raises PermissionError for any path outside the workspace folder.

</details>

## Summary

You built a coding agent from a model, five tools, and a loop, ran it on a failing
project, and verified its fix independently. You also saw two containment controls: a
workspace boundary and a step limit. Keep the `workspace-pricing` folder; Lab 04-02
customizes the agent in the same workspace.